# Milestone 11 — Laptop Review Usefulness Baseline

**Component:** Helpfulness-Guided Review Evidence Selection
with Minority Preservation

## Objective

Develop and evaluate a laptop review usefulness prediction
baseline using the cleaned Amazon Reviews 2023 laptop pilot.

## Experimental Protocol

1. Load the frozen Milestone 10 dataset and split.
2. Validate data and prevent target leakage.
3. Establish a DummyRegressor benchmark.
4. Train TF-IDF + Ridge models.
5. Compare text-only and text-plus-metadata features.
6. Select the configuration using validation NDCG@10.
7. Evaluate the chosen model on the untouched test set.
8. Generate product-level diagnostics.
9. Save model, predictions, metrics, and validation evidence.

The preliminary target is `log_helpful`.

No real C1 output is required for this milestone.

**Research limitation:** This is a controlled PP1 laptop
pilot. Full category-scale generalization and final
cross-category comparisons are outside this milestone.


## 1. Imports and Project Paths


In [1]:
from pathlib import Path
import json
import hashlib
import warnings

import joblib
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    ndcg_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data" / "pilot").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Component 2 project root was not found."
    )


ROOT = find_project_root(Path.cwd().resolve())

DATA_PATH = (
    ROOT / "data" / "pilot"
    / "laptop_pilot_cleaned.parquet"
)

SPLIT_PATH = (
    ROOT / "results" / "eda" / "laptop"
    / "laptop_chronological_split.csv"
)

RESULTS_DIR = (
    ROOT / "results" / "usefulness" / "laptop"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "log_helpful"

ALPHAS = [0.1, 1.0, 10.0, 100.0]
FEATURE_MODES = ["text_only", "text_metadata"]

print("Project root:", ROOT)
print("Laptop dataset:", DATA_PATH)
print("Split file:", SPLIT_PATH)
print("Results:", RESULTS_DIR)


Project root: D:\Research\Repository\J26-DS-349\component_2
Laptop dataset: D:\Research\Repository\J26-DS-349\component_2\data\pilot\laptop_pilot_cleaned.parquet
Split file: D:\Research\Repository\J26-DS-349\component_2\results\eda\laptop\laptop_chronological_split.csv
Results: D:\Research\Repository\J26-DS-349\component_2\results\usefulness\laptop


## 2. Load and Validate the Frozen Laptop Dataset


In [2]:
assert DATA_PATH.exists(), (
    "Milestone 10 cleaned laptop dataset is missing."
)

assert SPLIT_PATH.exists(), (
    "Milestone 10 chronological split is missing."
)

reviews = pd.read_parquet(DATA_PATH)

assignments = pd.read_csv(
    SPLIT_PATH,
    dtype={"review_id": str, "parent_asin": str},
)

required = {
    "review_id",
    "parent_asin",
    "text",
    "rating",
    "verified_purchase",
    "review_length_words",
    "age_days",
    "review_date",
    "helpful_vote",
    TARGET,
}

assert required.issubset(reviews.columns)

reviews["review_id"] = reviews["review_id"].astype(str)
reviews["parent_asin"] = reviews["parent_asin"].astype(str)

assert reviews["review_id"].is_unique
assert assignments["review_id"].is_unique

assert set(reviews["review_id"]) == set(
    assignments["review_id"]
)

df = reviews.merge(
    assignments[["review_id", "split"]],
    on="review_id",
    how="left",
    validate="one_to_one",
)

assert not df["split"].isna().any()

assert set(df["split"]) == {
    "train", "validation", "test"
}

df["review_date"] = pd.to_datetime(
    df["review_date"],
    utc=True,
)

df["text"] = df["text"].fillna("").astype(str)
df["verified_purchase"] = (
    df["verified_purchase"].astype(int)
)

df["rating"] = pd.to_numeric(df["rating"])
df["age_days"] = pd.to_numeric(df["age_days"])
df["review_length_words"] = pd.to_numeric(
    df["review_length_words"]
)

assert df[TARGET].notna().all()
assert (df[TARGET] >= 0).all()

assert df["parent_asin"].nunique() == 10
assert len(df) == 997

train = df[df["split"] == "train"].copy()
validation = df[df["split"] == "validation"].copy()
test = df[df["split"] == "test"].copy()

assert len(train) == 697
assert len(validation) == 150
assert len(test) == 150

assert not (
    set(train["review_id"])
    & set(validation["review_id"])
)

assert not (
    set(train["review_id"])
    & set(test["review_id"])
)

assert not (
    set(validation["review_id"])
    & set(test["review_id"])
)

print("Cleaned laptop reviews:", len(df))
print("Products:", df["parent_asin"].nunique())
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))


Cleaned laptop reviews: 997
Products: 10
Train: 697
Validation: 150
Test: 150


## 3. Chronological and Leakage Checks


In [3]:
chronology_rows = []

for product, group in df.groupby("parent_asin"):
    tr = group.loc[
        group["split"] == "train",
        "review_date",
    ]

    va = group.loc[
        group["split"] == "validation",
        "review_date",
    ]

    te = group.loc[
        group["split"] == "test",
        "review_date",
    ]

    chronological = bool(
        not tr.empty
        and not va.empty
        and not te.empty
        and tr.max() <= va.min()
        and va.max() <= te.min()
    )

    chronology_rows.append({
        "parent_asin": product,
        "chronology_valid": chronological,
        "train_count": len(tr),
        "validation_count": len(va),
        "test_count": len(te),
    })

chronology_df = pd.DataFrame(chronology_rows)

assert chronology_df["chronology_valid"].all()

# Only inputs available without using the target outcome.
PREDICTOR_COLUMNS = [
    "text",
    "rating",
    "verified_purchase",
    "review_length_words",
    "age_days",
]

FORBIDDEN_PREDICTORS = {
    "helpful_vote",
    "log_helpful",
    "raw_helpful",
    "product_relative",
    "age_adjusted",
    "split",
    "review_id",
}

assert not (
    set(PREDICTOR_COLUMNS)
    & FORBIDDEN_PREDICTORS
)

assert not df[PREDICTOR_COLUMNS].isna().any().any()

display(chronology_df)

print("Chronological split verification: PASS")
print("Target-derived feature exclusion: PASS")


,parent_asin,chronology_valid,train_count,validation_count,test_count
0,B009T3EYHO,True,70,15,15
1,B01K1IO3QW,True,69,15,15
2,B07GM2J11Q,True,69,15,15
3,B07N6S4SY1,True,70,15,15
4,B081V6W99V,True,70,15,15
5,B0869L1326,True,70,15,15
6,B08GG9QKX6,True,70,15,15
7,B096JQY4YR,True,69,15,15
8,B0C2ZMJW53,True,70,15,15
9,B0C7H1XD7V,True,70,15,15


Chronological split verification: PASS
Target-derived feature exclusion: PASS


## 4. Evaluation Functions


In [4]:
def product_ranking_metrics(frame):
    rows = []

    for product, group in frame.groupby("parent_asin"):
        true = group["actual"].to_numpy(dtype=float)
        predicted = group["predicted"].to_numpy(
            dtype=float
        )

        # An all-zero relevant set provides no positive
        # relevance signal for NDCG ranking evaluation.
        eligible = (
            len(group) >= 2
            and np.any(true > 0)
        )

        if eligible:
            ndcg5 = ndcg_score(
                true.reshape(1, -1),
                predicted.reshape(1, -1),
                k=min(5, len(group)),
            )

            ndcg10 = ndcg_score(
                true.reshape(1, -1),
                predicted.reshape(1, -1),
                k=min(10, len(group)),
            )
        else:
            ndcg5 = np.nan
            ndcg10 = np.nan

        if (
            len(group) >= 3
            and np.unique(true).size > 1
            and np.unique(predicted).size > 1
        ):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                rho = spearmanr(
                    true, predicted
                ).statistic
        else:
            rho = np.nan

        rows.append({
            "parent_asin": product,
            "review_count": len(group),
            "positive_relevance_count": int(
                np.sum(true > 0)
            ),
            "ndcg_eligible": bool(eligible),
            "ndcg_at_5": ndcg5,
            "ndcg_at_10": ndcg10,
            "spearman": rho,
            "mae": mean_absolute_error(
                true, predicted
            ),
            "rmse": np.sqrt(
                mean_squared_error(true, predicted)
            ),
        })

    return pd.DataFrame(rows)


def evaluate_predictions(frame, predicted, model_name):
    evaluated = frame[
        ["review_id", "parent_asin", "split"]
    ].copy()

    evaluated["actual"] = frame[
        TARGET
    ].to_numpy(dtype=float)

    evaluated["predicted"] = np.asarray(
        predicted, dtype=float
    )

    details = product_ranking_metrics(evaluated)

    true = evaluated["actual"].to_numpy()
    prediction = evaluated["predicted"].to_numpy()

    if (
        np.unique(true).size > 1
        and np.unique(prediction).size > 1
    ):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rho = spearmanr(
                true, prediction
            ).statistic
    else:
        rho = np.nan

    metrics = {
        "model": model_name,
        "review_count": len(evaluated),
        "product_count": details.shape[0],
        "ndcg_eligible_products": int(
            details["ndcg_eligible"].sum()
        ),
        "mae": float(
            mean_absolute_error(true, prediction)
        ),
        "rmse": float(np.sqrt(
            mean_squared_error(true, prediction)
        )),
        "spearman": float(rho),
        "ndcg_at_5": float(
            details["ndcg_at_5"].mean()
        ),
        "ndcg_at_10": float(
            details["ndcg_at_10"].mean()
        ),
    }

    return metrics, evaluated, details


## 5. Leakage-Safe Feature Pipeline


In [5]:
NUMERIC_COLUMNS = [
    "rating",
    "verified_purchase",
    "review_length_words",
    "age_days",
]


def make_model(mode, alpha):
    transformers = [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=15000,
                min_df=2,
                ngram_range=(1, 2),
                sublinear_tf=True,
                strip_accents="unicode",
            ),
            "text",
        )
    ]

    if mode == "text_metadata":
        transformers.append(
            (
                "numeric",
                StandardScaler(with_mean=False),
                NUMERIC_COLUMNS,
            )
        )

    features = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    return Pipeline([
        ("features", features),
        (
            "regressor",
            Ridge(
                alpha=float(alpha),
                solver="lsqr",
            ),
        ),
    ])


X_train = train[PREDICTOR_COLUMNS]
X_validation = validation[PREDICTOR_COLUMNS]
X_test = test[PREDICTOR_COLUMNS]

y_train = train[TARGET].to_numpy(dtype=float)
y_validation = validation[TARGET].to_numpy(
    dtype=float
)
y_test = test[TARGET].to_numpy(dtype=float)

print("Feature columns:", PREDICTOR_COLUMNS)
print("Feature fitting will use training reviews only.")


Feature columns: ['text', 'rating', 'verified_purchase', 'review_length_words', 'age_days']
Feature fitting will use training reviews only.


## 6. DummyRegressor Benchmark


In [6]:
dummy = DummyRegressor(strategy="mean")

dummy.fit(
    np.zeros((len(train), 1)),
    y_train,
)

dummy_validation_predictions = dummy.predict(
    np.zeros((len(validation), 1))
)

dummy_validation_metrics, _, _ = evaluate_predictions(
    validation,
    dummy_validation_predictions,
    "DummyRegressor",
)

display(pd.DataFrame([dummy_validation_metrics]))


,model,review_count,product_count,ndcg_eligible_products,mae,rmse,spearman,ndcg_at_5,ndcg_at_10
0,DummyRegressor,150,10,10,0.496693,0.554774,NaN,0.306499,0.44849


## 7. Train Ridge Candidates and Evaluate Validation


In [7]:
validation_results = []
trained_models = {}

for mode in FEATURE_MODES:
    for alpha in ALPHAS:
        name = f"{mode}_alpha_{alpha:g}"

        print("Training:", name)

        model = make_model(
            mode=mode,
            alpha=alpha,
        )

        # The entire feature pipeline is fitted on
        # training reviews only.
        model.fit(
            X_train,
            y_train,
        )

        predictions = model.predict(
            X_validation
        )

        metrics, _, _ = evaluate_predictions(
            validation,
            predictions,
            name,
        )

        metrics["feature_mode"] = mode
        metrics["alpha"] = float(alpha)

        validation_results.append(metrics)
        trained_models[name] = model


validation_table = pd.DataFrame(validation_results)

# NDCG@10 is the primary model-selection metric.
# Spearman and MAE are secondary tie-breakers.
validation_table = validation_table.sort_values(
    by=[
        "ndcg_at_10",
        "spearman",
        "mae",
    ],
    ascending=[
        False,
        False,
        True,
    ],
    na_position="last",
).reset_index(drop=True)

assert len(validation_table) == (
    len(FEATURE_MODES) * len(ALPHAS)
)

selected_row = validation_table.iloc[0]
SELECTED_MODEL_NAME = str(selected_row["model"])

selected_model = trained_models[
    SELECTED_MODEL_NAME
]

display(validation_table)

print("Selected model:", SELECTED_MODEL_NAME)
print(
    "Validation NDCG@10:",
    selected_row["ndcg_at_10"],
)


Training: text_only_alpha_0.1
Training: text_only_alpha_1
Training: text_only_alpha_10
Training: text_only_alpha_100
Training: text_metadata_alpha_0.1
Training: text_metadata_alpha_1
Training: text_metadata_alpha_10
Training: text_metadata_alpha_100


,model,review_count,product_count,ndcg_eligible_products,mae,rmse,spearman,ndcg_at_5,ndcg_at_10,feature_mode,alpha
0,text_metadata_alpha_10,150,10,10,0.376017,0.468806,0.488523,0.678112,0.757269,text_metadata,10.0
1,text_metadata_alpha_100,150,10,10,0.387047,0.468624,0.460905,0.667574,0.755010,text_metadata,100.0
2,text_metadata_alpha_1,150,10,10,0.388130,0.496842,0.473683,0.643226,0.740572,text_metadata,1.0
3,text_metadata_alpha_0.1,150,10,10,0.433252,0.559508,0.404439,0.603463,0.738579,text_metadata,0.1
4,text_only_alpha_10,150,10,10,0.446805,0.514923,0.454632,0.645540,0.723515,text_only,10.0
5,text_only_alpha_100,150,10,10,0.486662,0.544712,0.456506,0.619130,0.719825,text_only,100.0
6,text_only_alpha_0.1,150,10,10,0.475700,0.609374,0.370574,0.581283,0.699193,text_only,0.1
7,text_only_alpha_1,150,10,10,0.443209,0.543691,0.421116,0.566267,0.684602,text_only,1.0


Selected model: text_metadata_alpha_10
Validation NDCG@10: 0.7572690799486582


## 8. Final Held-Out Test Evaluation


In [8]:
# Do not retune the model after inspecting these results.

test_predictions = selected_model.predict(X_test)

test_metrics, test_output, test_product_metrics = (
    evaluate_predictions(
        test,
        test_predictions,
        SELECTED_MODEL_NAME,
    )
)

dummy_test_predictions = dummy.predict(
    np.zeros((len(test), 1))
)

dummy_test_metrics, _, _ = evaluate_predictions(
    test,
    dummy_test_predictions,
    "DummyRegressor",
)

test_comparison = pd.DataFrame([
    dummy_test_metrics,
    test_metrics,
])

display(test_comparison)
display(test_product_metrics)

print("Final selected model:", SELECTED_MODEL_NAME)
print("Held-out evaluation complete.")


,model,review_count,product_count,ndcg_eligible_products,mae,rmse,spearman,ndcg_at_5,ndcg_at_10
0,DummyRegressor,150,10,9,0.504586,0.547153,NaN,0.317270,0.452959
1,text_metadata_alpha_10,150,10,9,0.426670,0.502544,0.322268,0.552618,0.683366


,parent_asin,review_count,positive_relevance_count,ndcg_eligible,ndcg_at_5,ndcg_at_10,spearman,mae,rmse
0,B009T3EYHO,15,2,True,0.850345,0.850345,0.453921,0.357271,0.392839
1,B01K1IO3QW,15,3,True,0.469279,0.773746,0.308607,0.454702,0.510021
2,B07GM2J11Q,15,7,True,0.544726,0.704700,0.370375,0.414009,0.488364
3,B07N6S4SY1,15,3,True,0.513770,0.783896,0.312021,0.414453,0.490051
4,B081V6W99V,15,7,True,0.680970,0.759498,0.341970,0.556872,0.675441
5,B0869L1326,15,6,True,0.392015,0.595376,0.112082,0.493175,0.586614
6,B08GG9QKX6,15,0,False,NaN,NaN,NaN,0.431291,0.492888
7,B096JQY4YR,15,1,True,0.000000,0.000000,-0.371154,0.466867,0.520296
8,B0C2ZMJW53,15,8,True,0.566162,0.731962,0.490312,0.344894,0.418453
9,B0C7H1XD7V,15,6,True,0.956298,0.950767,0.780720,0.333171,0.378583


Final selected model: text_metadata_alpha_10
Held-out evaluation complete.


## 9. Review-Level Scoring and Provenance


In [9]:
# Scores for all reviews support later evidence selection.
# Training scores are in-sample and MUST NOT be reported
# as held-out performance.

all_predictions = selected_model.predict(
    df[PREDICTOR_COLUMNS]
)

all_review_scores = df[
    [
        "review_id",
        "parent_asin",
        "split",
    ]
].copy()

all_review_scores["predicted_log_helpful"] = (
    all_predictions
)

all_review_scores["predicted_usefulness"] = (
    np.maximum(all_predictions, 0.0)
)

provenance = {
    "train": "in_sample_training",
    "validation": "validation_used_for_model_selection",
    "test": "held_out_test",
}

all_review_scores["prediction_provenance"] = (
    all_review_scores["split"].map(provenance)
)

assert all_review_scores["review_id"].is_unique
assert len(all_review_scores) == len(df)

assert np.isfinite(
    all_review_scores["predicted_log_helpful"]
).all()

display(all_review_scores.head())


,review_id,parent_asin,split,predicted_log_helpful,predicted_usefulness,prediction_provenance
0,fc7d6c75439496a7,B009T3EYHO,train,9.462160,9.462160,in_sample_training
1,03ca84f86251d5e3,B009T3EYHO,train,2.678191,2.678191,in_sample_training
2,a323d6369f2f0979,B009T3EYHO,train,1.873454,1.873454,in_sample_training
3,4967d597c7dce12e,B009T3EYHO,train,1.377946,1.377946,in_sample_training
4,2dc7bb4cf2705ccd,B009T3EYHO,train,0.844473,0.844473,in_sample_training


## 10. Save Reproducible Evidence


In [10]:
MODEL_PATH = RESULTS_DIR / "laptop_ridge_model.joblib"

VALIDATION_PATH = (
    RESULTS_DIR / "laptop_validation_model_selection.csv"
)

TEST_METRICS_PATH = (
    RESULTS_DIR / "laptop_test_metrics.csv"
)

PRODUCT_METRICS_PATH = (
    RESULTS_DIR / "laptop_product_test_metrics.csv"
)

TEST_PREDICTIONS_PATH = (
    RESULTS_DIR / "laptop_test_predictions.csv"
)

ALL_PREDICTIONS_PATH = (
    RESULTS_DIR / "laptop_all_review_predictions.csv"
)

CONFIG_PATH = RESULTS_DIR / "laptop_model_config.json"
AUDIT_PATH = RESULTS_DIR / "laptop_split_audit.csv"

joblib.dump(
    selected_model,
    MODEL_PATH,
)

validation_table.to_csv(
    VALIDATION_PATH,
    index=False,
)

test_comparison.to_csv(
    TEST_METRICS_PATH,
    index=False,
)

test_product_metrics.to_csv(
    PRODUCT_METRICS_PATH,
    index=False,
)

test_output.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)

all_review_scores.to_csv(
    ALL_PREDICTIONS_PATH,
    index=False,
)

chronology_df.to_csv(
    AUDIT_PATH,
    index=False,
)

config = {
    "milestone": 11,
    "category": "laptop",
    "dataset": str(DATA_PATH.relative_to(ROOT)),
    "split_file": str(SPLIT_PATH.relative_to(ROOT)),
    "target": TARGET,
    "candidate_alphas": ALPHAS,
    "feature_modes": FEATURE_MODES,
    "selected_model": SELECTED_MODEL_NAME,
    "selected_alpha": float(selected_row["alpha"]),
    "selected_feature_mode": str(
        selected_row["feature_mode"]
    ),
    "model_selection_metric": "validation_ndcg_at_10",
    "test_policy": "held_out_no_retuning",
    "numeric_features": NUMERIC_COLUMNS,
    "forbidden_predictors": sorted(FORBIDDEN_PREDICTORS),
    "all_review_prediction_warning": (
        "Training review predictions are in-sample; "
        "validation reviews were used for model selection. "
        "Only test predictions are held out."
    ),
    "ranking_metric_note": (
        "Per-product NDCG excludes products with "
        "no positive relevance labels and reports "
        "the eligible product count."
    ),
    "test_metrics": test_metrics,
    "dummy_test_metrics": dummy_test_metrics,
}

CONFIG_PATH.write_text(
    json.dumps(
        config,
        indent=2,
        allow_nan=True,
    ),
    encoding="utf-8",
)

print("Model saved:", MODEL_PATH)
print("Validation evidence:", VALIDATION_PATH)
print("Test evidence:", TEST_METRICS_PATH)
print("All review scores:", ALL_PREDICTIONS_PATH)


Model saved: D:\Research\Repository\J26-DS-349\component_2\results\usefulness\laptop\laptop_ridge_model.joblib
Validation evidence: D:\Research\Repository\J26-DS-349\component_2\results\usefulness\laptop\laptop_validation_model_selection.csv
Test evidence: D:\Research\Repository\J26-DS-349\component_2\results\usefulness\laptop\laptop_test_metrics.csv
All review scores: D:\Research\Repository\J26-DS-349\component_2\results\usefulness\laptop\laptop_all_review_predictions.csv


## 11. Final Validation and Determinism


In [11]:
# Load the serialized model and verify predictions.
reloaded_model = joblib.load(MODEL_PATH)

reloaded_predictions = reloaded_model.predict(
    X_test
)

np.testing.assert_allclose(
    test_predictions,
    reloaded_predictions,
    rtol=1e-9,
    atol=1e-9,
)

assert len(train) + len(validation) + len(test) == 997

assert df["parent_asin"].nunique() == 10

assert chronology_df["chronology_valid"].all()

assert len(validation_table) == 8

assert len(test_output) == 150

assert len(all_review_scores) == 997

assert all_review_scores["review_id"].is_unique

assert set(all_review_scores["split"]) == {
    "train", "validation", "test"
}

assert set(
    all_review_scores["prediction_provenance"]
) == {
    "in_sample_training",
    "validation_used_for_model_selection",
    "held_out_test",
}

assert np.isfinite(test_predictions).all()

assert all(path.exists() for path in [
    MODEL_PATH,
    VALIDATION_PATH,
    TEST_METRICS_PATH,
    PRODUCT_METRICS_PATH,
    TEST_PREDICTIONS_PATH,
    ALL_PREDICTIONS_PATH,
    CONFIG_PATH,
    AUDIT_PATH,
])

print()
print("Laptop Function 1 baseline validation passed.")
print()
print("Selected model:", SELECTED_MODEL_NAME)
print("Target:", TARGET)
print("Train reviews:", len(train))
print("Validation reviews:", len(validation))
print("Test reviews:", len(test))
print()
print("Test MAE:", round(test_metrics["mae"], 4))
print("Test RMSE:", round(test_metrics["rmse"], 4))
print(
    "Test Spearman:",
    round(test_metrics["spearman"], 4),
)
print(
    "Test NDCG@5:",
    round(test_metrics["ndcg_at_5"], 4),
)
print(
    "Test NDCG@10:",
    round(test_metrics["ndcg_at_10"], 4),
)
print(
    "NDCG-eligible products:",
    test_metrics["ndcg_eligible_products"],
)



Laptop Function 1 baseline validation passed.

Selected model: text_metadata_alpha_10
Target: log_helpful
Train reviews: 697
Validation reviews: 150
Test reviews: 150

Test MAE: 0.4267
Test RMSE: 0.5025
Test Spearman: 0.3223
Test NDCG@5: 0.5526
Test NDCG@10: 0.6834
NDCG-eligible products: 9


## 12. Research Limitations and Next Milestone

- The experiment uses 997 reviews from ten laptop products.
- `log_helpful` is a noisy observational proxy for usefulness.
- Zero helpful votes may reflect low exposure rather than poor content.
- Product popularity and temporal sampling may introduce bias.
- Held-out test metrics are distinct from all-review inference scores.
- The selected model is laptop-specific at this stage.
- A direct smartphone-versus-laptop comparison requires aligning
  evaluation policies across both categories.
- Training-review in-sample predictions should not be used as
  unbiased downstream selector-evaluation evidence.

**Next milestone:** Laptop Semantic Representation and Function 2
Evidence Selection.
